In [1]:
import os
import json
import chromadb
import networkx as nx
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from chromadb.api.types import EmbeddingFunction

# ----------- Configuration -----------
FILE_PATH = "rag_interpretations.txt"   # Now stores *interpretations*, not raw responses
GRAPH_PATH = "graph_rag.gexf"

# ----------- Document Handling -----------
def load_documents(file_path=FILE_PATH):
    if not os.path.exists(file_path):
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        docs = [line.strip() for line in f.readlines() if line.strip()]
    return docs

def save_interpretation_to_file(text, file_path=FILE_PATH):
    """Store only the interpreted meaning of the conversation."""
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(text + "\n")


c:\Users\suraj\LLM_projects\RAG_memory\LLM_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ----------- Embedding Function -----------
class MiniLMEmbeddings(EmbeddingFunction):
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")

    def __call__(self, texts):
        return self.model.encode(texts).tolist()

    def name(self):
        return "all-MiniLM-L6-v2"

embedding_function = MiniLMEmbeddings()

# ----------- ChromaDB Setup -----------
client = chromadb.PersistentClient(path="./rag_db")

try:
    client.delete_collection("rag_collection")
except:
    pass

collection = client.get_or_create_collection(
    name="rag_collection",
    embedding_function=embedding_function
)
# ----------- Load LLM Model -----------
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype="auto")
llm = pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False)


`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda:0


In [3]:

# ----------- GraphRAG Setup -----------
graph = nx.DiGraph()
if os.path.exists(GRAPH_PATH):
    graph = nx.read_gexf(GRAPH_PATH)

def add_to_graph(text):
    """Extract entities and relationships from text and add to GraphRAG graph."""
    prompt = f"""
Extract key entities and their relationships from the following text 
as a JSON list of triples formatted like this:
[{{"entity1": "A", "relation": "relates_to", "entity2": "B"}}]

Text:
{text}

Output JSON:
    """
    triples_json = llm(prompt, max_new_tokens=200, do_sample=False)[0]["generated_text"].strip()
    try:
        triples = json.loads(triples_json)
        for t in triples:
            graph.add_edge(t["entity1"], t["entity2"], relation=t["relation"])
        nx.write_gexf(graph, GRAPH_PATH)
    except Exception as e:
        print("⚠️ Could not parse graph triples:", e)

# ----------- Interpretation Update -----------
def update_interpretation():
    """Generate a high-level interpretation of the current RAG + Graph state and store it."""
    # Gather available text context (from docs and graph)
    documents = load_documents()
    context_text = "\n".join(documents)

    graph_summary_prompt = f"""
You are an AI assistant reflecting on your stored knowledge graph and text memory.
Given the following contextual text (previous interpretations) and relationships, 
produce a concise interpretation (1–3 sentences) summarizing the key meaning or 
insight contained within.

Text context:
{context_text}

Interpretation:
    """

    interpretation = llm(graph_summary_prompt, max_new_tokens=200, do_sample=True)[0]["generated_text"].strip()
    save_interpretation_to_file(interpretation)
    print("🧠 Interpretation updated:", interpretation[:100], "...")
    return interpretation


In [4]:

# ----------- RAG Query Function -----------
def rag_query(question, k=3):
    results = collection.query(query_texts=[question], n_results=k)
    context = "\n".join(results["documents"][0]) if results["documents"] else ""

    prompt = f"""
Answer the following question concisely based on the provided context.
If the context is insufficient, provide a related or thoughtful response.
Output only the final answer.

Context:
{context}

Question:
{question}

Answer:
    """

    answer = llm(prompt, max_new_tokens=200, do_sample=True)[0]["generated_text"].strip()

    # ✅ Add full answer to graph memory
    add_to_graph(answer)

    # ✅ Update global interpretation file (reflective understanding)
    update_interpretation()

    return answer


In [8]:

# ----------- Test Query -----------
if __name__ == "__main__":
    print("\nRAG Response:")
    print(rag_query("Can you tell something based on our past discussions?"))



RAG Response:


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


🧠 Interpretation updated: Effective leadership combines clear communication, adaptability, empathy, and accountability to buil ...
I'm sorry, but I don't see any prior discussions to base an answer off of. Please provide more details or context for me to respond appropriately. 

Related thought: Effective communication often relies on building upon previous conversations and shared experiences. Without that foundation, it's difficult to provide meaningful insights into what has been discussed previously. To give a helpful response, we would need information from those past exchanges.
